# Option 1b – Read the on-prem lake from a notebook, outside the SQL engine

DuckDB in this container talks **directly** to the tunnel (`tunnel-service:8181` = on-prem Iceberg catalog, `:9000` = on-prem object store).
The sensitive rows never pass through a Snowflake SQL query, so there is **no persisted query result and nothing for `RESULT_SCAN`**.

Remaining exposure: notebook memory while the kernel runs, and anything you *print* (cell output is saved with the notebook). This notebook prints aggregates only.

Close the tunnel (`pixi run tunnel-down`) and re-run cell 3 – the read fails.

In [ ]:
# 1. DuckDB (installed via the notebook EAI if the runtime does not ship it)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])
    import duckdb

from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Internal DNS of the tunnel service (only reachable from inside Snowflake, same owner role)
rows = session.sql("SHOW SERVICES LIKE 'TUNNEL_SERVICE' IN SCHEMA SOVEREIGNTY_DEMO.OPT1_TUNNEL").collect()
TUNNEL = rows[0]['dns_name']
print('tunnel service:', TUNNEL, '| status:', rows[0]['status'])

In [ ]:
# 2. Point DuckDB at the tunnel ports. Iceberg metadata references the on-prem host 'minio:9000';
#    the S3 secret's ENDPOINT override sends those data-file reads to the tunnel instead.
con = duckdb.connect(':memory:')
con.execute('INSTALL iceberg; LOAD iceberg; INSTALL httpfs; LOAD httpfs;')
con.execute(f"""CREATE OR REPLACE SECRET onprem_s3 (TYPE S3, KEY_ID 'admin', SECRET 'password',
                ENDPOINT '{TUNNEL}:9000', USE_SSL false, URL_STYLE 'path', REGION 'us-east-1')""")
con.execute("CREATE OR REPLACE SECRET onprem_rest (TYPE ICEBERG, TOKEN 'dummy')")
print('DuckDB configured against the tunnel')

In [ ]:
# 3. Live read through the tunnel + join with the public half from Snowflake. Aggregates only.
public = session.table('SOVEREIGNTY_DEMO.OPT1_TUNNEL.EMPLOYEES_PUBLIC').to_pandas()   # non-sensitive half
con.register('employees_public', public)
try:
    try:
        con.execute('DETACH demo')
    except Exception:
        pass
    con.execute(f"""ATTACH 'demo' AS demo (TYPE ICEBERG, ENDPOINT 'http://{TUNNEL}:8181/catalog/',
                    SECRET onprem_rest, ACCESS_DELEGATION_MODE 'none')""")
    summary = con.execute("""
        SELECT p.DEPARTMENT AS department, COUNT(*) AS headcount, ROUND(AVG(s.salary)) AS avg_salary
        FROM employees_public p
        JOIN demo.demo.employees_sensitive s ON s.employee_id = p.EMPLOYEE_ID
        GROUP BY 1 ORDER BY 1""").fetchdf()
    print('Read live through the tunnel (computed in this container, not by Snowflake SQL):')
    print(summary.to_string(index=False))
except Exception as e:
    raise RuntimeError('Tunnel closed or unreachable -> no route to the on-prem data') from e

In [ ]:
# 4. Proof: no SQL query in this session touched the sensitive table -> nothing for RESULT_SCAN.
hits = session.sql("""
    SELECT COUNT(*) AS n FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 1000))
    WHERE query_text ILIKE '%employees_sensitive%' AND query_text NOT ILIKE '%QUERY_HISTORY_BY_SESSION%'
""").collect()[0]['N']
print('Snowflake SQL queries in this session that touched the sensitive data:', hits)

In [ ]:
# 5. Performance (optional): time the same join 10 times in this container. Only timings are written to
#    Snowflake (OPT1_TUNNEL.BENCH_1B) so `bench/run_all.py` can pick them up - no employee data.
import time, statistics, json
def _ms(f):
    t = time.perf_counter(); f(); return (time.perf_counter() - t) * 1000
JOIN_SQL = """SELECT p.DEPARTMENT, COUNT(*), ROUND(AVG(s.salary)) FROM employees_public p
              JOIN demo.demo.employees_sensitive s ON s.employee_id = p.EMPLOYEE_ID GROUP BY 1"""
pull = [_ms(lambda: session.table('SOVEREIGNTY_DEMO.OPT1_TUNNEL.EMPLOYEES_PUBLIC').to_pandas()) for _ in range(5)]
join = [_ms(lambda: con.execute(JOIN_SQL).fetchall()) for _ in range(10)]
read = [_ms(lambda: con.execute('SELECT COUNT(*), SUM(salary) FROM demo.demo.employees_sensitive').fetchall())
        for _ in range(10)]
res = {k: {'p50': round(statistics.median(v)), 'max': round(max(v)), 'n': len(v)}
       for k, v in (('pull_public_snowpark', pull), ('duckdb_join_through_tunnel', join), ('duckdb_read_200', read))}
print(json.dumps(res, indent=2))
session.sql("CREATE TABLE IF NOT EXISTS SOVEREIGNTY_DEMO.OPT1_TUNNEL.BENCH_1B (ts TIMESTAMP_LTZ, result VARIANT)").collect()
session.sql("INSERT INTO SOVEREIGNTY_DEMO.OPT1_TUNNEL.BENCH_1B SELECT CURRENT_TIMESTAMP(), PARSE_JSON(?)",
            params=[json.dumps(res)]).collect()